## 🌳 LeetCode 199: Binary Tree Right Side View
---

<div style="padding: 15px; border-left: 8px solid #1a8a5a; background-color: #e8f5e9; color: #1b5e20; border-radius: 4px;">
    <strong>The Core Insight:</strong> Two equivalent approaches:
    (1) BFS — take the LAST node at each level.
    (2) DFS right-first — the FIRST node encountered at each depth is the rightmost.
    BFS is more intuitive. DFS is more space-efficient for balanced trees.
</div>

### 📋 Problem

Imagine standing to the right of a tree and looking left.
Return the values of the nodes you can see (one per level).

```
Input:  [1, 2, 3, null, 5, null, 4]
Output: [1, 3, 4]

    1         ← see 1
   / \
  2   3       ← see 3 (rightmost)
   \   \
    5   4     ← see 4 (rightmost)
```

**Constraints:** 0 ≤ nodes ≤ 100 | -100 ≤ Node.val ≤ 100


### 🧠 Choose Your Mental Model

| Model | The "Story" | Mechanism |
|-------|-------------|-----------|
| **Building from the Right** | "I'm standing at the right end of the building. On each floor, the rightmost office's window is the only one I can see." | BFS: take `level[-1]` (last per level) |
| **Right-First Explorer** | "I always go right before left. The FIRST node I see at any depth is the rightmost one at that depth — nothing to its right was visited yet." | DFS right-first: `depth == len(result)` → append |
| **Obstructed View** | "A node is hidden if there's another node at the same depth to its right — it's blocked from view." | Equivalent to: rightmost node at each depth |


In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals):
    """Build tree from level-order list (None = missing node)."""
    if not vals or vals[0] is None:
        return None
    from collections import deque
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

print("TreeNode ready.")

def rightSideView_brute(root: TreeNode) -> list[int]:
    """
    Collect full level order, take last element of each level.
    Time: O(n)  |  Space: O(n)
    Correct but builds unnecessary intermediate structure (all levels).
    """
    if not root:
        return []

    from collections import deque
    result = []
    queue = deque([root])

    while queue:
        level_size = len(queue)
        level = []
        for _ in range(level_size):
            node = queue.popleft()
            level.append(node.val)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level[-1])   # only keep rightmost

    return result


# Test
root = build_tree([1, 2, 3, None, 5, None, 4])
print("Brute force:", rightSideView_brute(root))   # [1, 3, 4]


## 🔬 Complexity Analysis

<div style="padding: 15px; border-left: 8px solid #1a8a5a; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>The Core Insight:</strong> Both BFS and DFS are O(n) time.
    The tradeoff is space: BFS is O(w) (max level width), DFS is O(h) (tree height).
    For a balanced tree: O(n) vs O(log n) — DFS wins on space.
    For a skewed tree: O(1) vs O(n) — BFS wins on space.
</div>

---

### The DFS Right-Side-View Insight

When traversing right-first DFS:
- The FIRST time we reach depth d, it's the rightmost node at that depth
- "First time at depth d" = `depth == len(result)` (result has entries 0..d-1)

```
depth=0: result=[], append 1  → result=[1]
depth=1: result=[1], we go RIGHT first → reach 3 first → append 3 → result=[1,3]
depth=1: then go left → reach 2, but depth 1 already in result → skip
depth=2: result=[1,3], right subtree of 3 has 4 → append 4 → result=[1,3,4]
depth=2: right subtree of 2 has 5 → but depth 2 already in result → skip
```

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| BFS full level order → take last | O(n) | O(w) | Builds all levels |
| **BFS — capture on last node** | **O(n)** | **O(w)** | Optimal BFS |
| DFS right-first | O(n) | O(h) | Better space for balanced trees |

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Default answer:</strong> BFS (more intuitive). Mention DFS right-first as the follow-up.
</div>


In [ ]:
from collections import deque

def rightSideView_bfs(root: TreeNode) -> list[int]:
    """
    BFS: capture the LAST node at each level.
    Time: O(n)  |  Space: O(w) — max level width
    The critical condition: i == level_size - 1 (last node at this level).
    """
    if not root:
        return []

    result = []
    queue = deque([root])

    while queue:
        level_size = len(queue)
        for i in range(level_size):
            node = queue.popleft()
            if i == level_size - 1:       # last node at this level — visible from right
                result.append(node.val)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)

    return result


def rightSideView_dfs(root: TreeNode) -> list[int]:
    """
    DFS right-first: first node seen at each depth is the rightmost.
    Time: O(n)  |  Space: O(h) — call stack depth
    More space-efficient for balanced trees.
    """
    result = []

    def dfs(node, depth):
        if not node:
            return
        if depth == len(result):         # first time at this depth = rightmost node
            result.append(node.val)
        dfs(node.right, depth + 1)       # ← RIGHT first (critical)
        dfs(node.left,  depth + 1)

    dfs(root, 0)
    return result


# Tests
root = build_tree([1, 2, 3, None, 5, None, 4])
print("BFS:", rightSideView_bfs(root))    # [1, 3, 4]
print("DFS:", rightSideView_dfs(root))    # [1, 3, 4]

# Left side view = same code, DFS LEFT-first OR BFS take first element per level
def leftSideView(root):
    result = []
    def dfs(node, depth):
        if not node: return
        if depth == len(result):
            result.append(node.val)
        dfs(node.left, depth + 1)    # LEFT first for left side view
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return result

print("Left:", leftSideView(root))   # [1, 2, 5]


## 🎤 5 Interview Q&A

### Q1: In the DFS solution, why do we traverse right BEFORE left?

**Answer:** We want the FIRST node we see at each depth to be the rightmost one.
If we go right first, at any depth d, the first node we encounter is the one farthest to the right
(no unvisited nodes to its right). The condition `depth == len(result)` catches only
the first visit to each depth — so it captures the rightmost node.
If we went left first, we'd get the left side view instead.

---

### Q2: How would you modify this to get the LEFT side view?

**Answer:** Two changes for DFS: traverse LEFT before right (swap the two recursive calls).
For BFS: take the FIRST element of each level instead of the last:
```python
if i == 0:           # first node = leftmost
    result.append(node.val)
```

---

### Q3: What is the space complexity of DFS vs BFS for a balanced tree of n nodes?

**Answer:**
- DFS: O(h) = O(log n) — call stack depth equals tree height
- BFS: O(w) = O(n) — worst case, the leaf level has n/2 nodes in the queue

So DFS is more space-efficient for balanced trees. For a skewed tree (all nodes to the right),
DFS space becomes O(n) while BFS stays O(1). The choice depends on expected tree shape.

---

### Q4: What if a node exists only on the left subtree — does the right side view capture it?

**Answer:** Yes, if it's the only node at its depth. Example: root with only a left child.
The left child IS the rightmost node at depth 1, so it appears in the right side view.
"Right side view" means the last node at each depth in BFS — if the only node at a depth
is on the left, it's still visible from the right.

---

### Q5: How does this relate to the problem of finding the "boundary" of a binary tree?

**Answer:** The right side view is the right boundary of the tree.
The full boundary = left boundary (top-down, left-first, no leaves) +
all leaves (left to right) + right boundary (bottom-up, right-first, no leaves).
The right side view is a simpler version: just the rightmost node at each level
including the root and all rightmost nodes, collected top-down.


## 💼 The Citi Narrative

**Context:** Citi's network topology was a tree: Core Switch → Distribution Switches → Access Switches → Servers.
The "right side view" of this tree represented the edge-most devices at each tier —
the servers and switches at the outermost boundary of the network.

**Practical Application:** Security team needed to identify all "edge-facing" devices —
those at the outermost position in each network tier. These required the strictest firewall rules.
The right side view algorithm identified exactly these devices: one per tier, the rightmost
(highest-index) device at each network depth.

**Extended Version:** Instead of just the rightmost value, the team needed all boundary devices
(left boundary + all leaves + right boundary). This maps to the "Binary Tree Boundary" problem —
but started with the right side view as the building block.

**Real data:** The network tree had:
- Depth 0: 1 core switch
- Depth 1: 8 distribution switches
- Depth 2: 64 access switches
- Depth 3: ~6,000 server ports

Right side view returned 4 values (one per depth). Security reviewed these 4 "most exposed"
devices as the starting point for network audit.

**Interview Line:** *"At Citi the network was a tree. The security audit started with
the 'rightmost device at each tier' — that's the right side view. BFS, take last element
per level. 10 lines of code, then extend to full boundary traversal."*


In [ ]:
# ── PRACTICE: Find the Largest Value in Each Tree Row (LC #515) ────────────
# Given the root of a binary tree, return the largest value in each row.
# Example: [1,3,2,5,3,null,9] → [1,3,9]
# Hint: same BFS skeleton, but instead of last value, track max value per level.

def largestValues(root: TreeNode) -> list[int]:
    # Your solution here
    pass

root = build_tree([1, 3, 2, 5, 3, None, 9])
print(largestValues(root))   # Expected: [1, 3, 9]

# Solution:
def largestValues_sol(root):
    if not root: return []
    from collections import deque
    result, queue = [], deque([root])
    while queue:
        level_size = len(queue)
        level_max = float('-inf')
        for _ in range(level_size):
            node = queue.popleft()
            level_max = max(level_max, node.val)
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level_max)
    return result

print("Largest per row:", largestValues_sol(build_tree([1, 3, 2, 5, 3, None, 9])))  # [1,3,9]


## 🎯 Summary: Key Takeaways

### The Two Patterns

**BFS (intuitive):**
```python
if i == level_size - 1:    # last node at this level
    result.append(node.val)
```

**DFS right-first (space-efficient for balanced trees):**
```python
if depth == len(result):   # first time at this depth = rightmost
    result.append(node.val)
dfs(node.right, depth+1)   # RIGHT before LEFT
dfs(node.left,  depth+1)
```

### Variants Using the Same Skeleton
| Problem | Change |
|---------|--------|
| Right side view (this) | BFS last OR DFS right-first |
| Left side view | BFS first OR DFS left-first |
| Largest value per row | BFS: track max per level |
| Average per level | BFS: sum/count per level |
| Level order traversal | BFS: collect all per level |

### Interview Confidence Checklist
- [ ] Can write BFS solution from memory in 3 minutes
- [ ] Can write DFS right-first solution from memory
- [ ] Can explain why right-first DFS captures rightmost node
- [ ] Can extend to left side view (just swap left/right)
- [ ] Can name the Citi context: edge-facing network devices

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀
